In [1]:
from pathlib import Path
import subprocess
import sys

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

from src.utils.paths import get_data_dir, get_landmark_dir, get_processed_dir, get_teeth3ds_dir

DATA_DIR = get_data_dir()
RAW_DIR = get_teeth3ds_dir()
LANDMARK_DIR = get_landmark_dir()
OUT_DIR = get_processed_dir()

print("PROJECT:", PROJECT)
print("DATA_DIR:", DATA_DIR)
print("RAW_DIR:", RAW_DIR)
print("LANDMARK_DIR:", LANDMARK_DIR)
print("OUT_DIR:", OUT_DIR)

PROJECT: /Users/nourlachtar/projets/projet/Orthotwin3D
DATA_DIR: /Users/nourlachtar/Documents/OrthoTwin3D/data
RAW_DIR: /Users/nourlachtar/Documents/OrthoTwin3D/data/raw/Teeth3DS
LANDMARK_DIR: /Users/nourlachtar/Documents/OrthoTwin3D/data/raw/Teeth3DSLandmarks
OUT_DIR: /Users/nourlachtar/Documents/OrthoTwin3D/data/processed


## Parametres

In [22]:
NUM_POINTS = 8192
SAMPLING = "random"
SEED = 42
LIMIT = 10           # mets None pour tout traiter
CURVATURE = True    # True pour ajouter la feature curvature

## Lancer la preparation

In [23]:
cmd = [
    sys.executable,
    str(PROJECT / "scripts/prepare_data.py"),
    "--raw_dir", str(RAW_DIR),
    "--landmark_dir", str(LANDMARK_DIR),
    "--out_dir", str(OUT_DIR),
    "--num_points", str(NUM_POINTS),
    "--sampling", SAMPLING,
    "--seed", str(SEED),
]

if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]
if CURVATURE:
    cmd += ["--curvature"]

print(" ".join(cmd))
result = subprocess.run(cmd, cwd=PROJECT, text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
result.check_returncode()

/Users/nourlachtar/projets/projet/Orthotwin3D/.venv/bin/python /Users/nourlachtar/projets/projet/Orthotwin3D/scripts/prepare_data.py --raw_dir /Users/nourlachtar/Documents/OrthoTwin3D/data/raw/Teeth3DS --landmark_dir /Users/nourlachtar/Documents/OrthoTwin3D/data/raw/Teeth3DSLandmarks --out_dir /Users/nourlachtar/Documents/OrthoTwin3D/data/processed --num_points 8192 --sampling random --seed 42 --limit 10 --curvature
INFO | Discovered 10 scans
INFO | Wrote 10 samples to /Users/nourlachtar/Documents/OrthoTwin3D/data/processed



## Fichiers produits

In [24]:
files = sorted(OUT_DIR.glob("*.pt"))
print("processed_dir:", OUT_DIR)
print("num files:", len(files))
files[:10]

processed_dir: /Users/nourlachtar/Documents/OrthoTwin3D/data/processed
num files: 10


[PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/00OMSZGW_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/01328DDN_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/0132CR0A_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/01343APK_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/01346914_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/013475VT_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/013FHA7K_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/013FWKMZ_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/013H18FF_lower.pt'),
 PosixPath('/Users/nourlachtar/Documents/OrthoTwin3D/data/processed/013H3Y8H_lower.pt')]

In [51]:
from pprint import pprint

from src.utils.io import load_processed_sample

files = sorted(OUT_DIR.glob("*.pt"))
if not files:
    raise FileNotFoundError(f"Aucun fichier .pt trouve dans {OUT_DIR}. Lance la preparation avant cette cellule.")

example_path = None
sample = None
for path in files:
    current = load_processed_sample(path)
    if current.get("landmarks_raw"):
        example_path = path
        sample = current
        break

if sample is None:
    raise FileNotFoundError(f"Aucun sample avec landmarks_raw trouve dans {OUT_DIR}. Augmente LIMIT ou mets LIMIT=None, puis relance la preparation.")

def section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


section("SAMPLE")
print("path:", example_path)
print("scan_id:", sample["scan_id"])
print("patient_id:", sample["patient_id"])
print("jaw:", sample["jaw"])
print("num keys:", len(sample))
print("keys:", list(sample.keys()))
print()


section("TENSORS")
tensor_keys = ["pos_raw", "pos", "normal", "y_binary", "y_fdi", "y_fdi_class", "y_instance", "source_indices", "center"]
for key in tensor_keys:
    value = sample.get(key)
    if value is None:
        print(f"{key:<16} None")
    else:
        values = value.reshape(-1)[:5].tolist()
        print(f"{key:<16} shape={str(tuple(value.shape)):<14} dtype={str(value.dtype):<13} values={values}")
    print()


section("SCALARS / MAPPINGS")
print("normal_source:", sample["normal_source"])
print()
print("curvature:", "None" if sample["curvature"] is None else f"shape={tuple(sample['curvature'].shape)}")
print()
print("scale:", sample["scale"])
print()
print("fdi_to_class len:", len(sample["fdi_to_class"]), "example:", dict(list(sample["fdi_to_class"].items())[:5]))
print()
print("class_to_fdi len:", len(sample["class_to_fdi"]), "example:", dict(list(sample["class_to_fdi"].items())[:5]))


section("LANDMARKS")
landmarks_raw = sample.get("landmarks_raw") or {}
landmarks_norm = sample.get("landmarks_norm") or {}
by_tooth = landmarks_raw.get("by_tooth", {})
print("landmarks_raw keys:", list(landmarks_raw.keys()))
print("num teeth with landmarks:", len(by_tooth))
print("teeth example list:", list(by_tooth.keys())[:10])
print()

if by_tooth:
    tooth_key = next(iter(by_tooth))
    class_key = next(iter(by_tooth[tooth_key]))
    raw_record = by_tooth[tooth_key][class_key][0]
    norm_record = landmarks_norm["by_tooth"][tooth_key][class_key][0]
    print(f"example tooth={tooth_key}, landmark_class={class_key}")
    print("raw:")
    pprint(raw_record)
    print()
    print("norm:")
    pprint(norm_record)
    print()

landmark_to_tooth = sample.get("landmark_to_tooth") or []
print("landmark_to_tooth len:", len(landmark_to_tooth))
if landmark_to_tooth:
    print("landmark_to_tooth example:")
    pprint(landmark_to_tooth[0])
print()


section("TOOTH CENTERS")
for key in ["tooth_centers_raw", "tooth_centers_norm"]:
    centers = sample[key]
    first_tooth = next(iter(centers), None)
    print(key, "len:", len(centers), "teeth:", list(centers.keys())[:10])
    if first_tooth is not None:
        print(f"example {first_tooth}:", centers[first_tooth])
    print()





SAMPLE
path: /Users/nourlachtar/Documents/OrthoTwin3D/data/processed/0140W3ND_lower.pt
scan_id: 0140W3ND_lower
patient_id: 0140W3ND
jaw: lower
num keys: 22
keys: ['scan_id', 'patient_id', 'jaw', 'pos_raw', 'pos', 'normal', 'normal_source', 'curvature', 'y_binary', 'y_fdi', 'y_fdi_class', 'y_instance', 'fdi_to_class', 'class_to_fdi', 'source_indices', 'landmarks_raw', 'landmarks_norm', 'landmark_to_tooth', 'tooth_centers_raw', 'tooth_centers_norm', 'center', 'scale']


TENSORS
pos_raw          shape=(8192, 3)      dtype=torch.float32 values=[-13.669410705566406, -1.037185549736023, -91.19403076171875, 23.884876251220703, -1.8309537172317505]

pos              shape=(8192, 3)      dtype=torch.float32 values=[-0.2691909968852997, -0.0066255489364266396, -0.019585713744163513, 0.3745054602622986, -0.02023107185959816]

normal           shape=(8192, 3)      dtype=torch.float32 values=[0.9758033752441406, 0.054583337157964706, 0.21172703802585602, 0.7416398525238037, -0.2617543339729309]

y

## Attributs du sample

| Attribut | Definition courte |
|---|---|
| `scan_id` | Identifiant du scan source. |
| `patient_id` | Identifiant patient deduit du nom du scan. |
| `jaw` | Arcade du scan: `upper` ou `lower`. |
| `pos_raw` | Coordonnees XYZ echantillonnees dans le repere original du mesh. |
| `pos` | Coordonnees XYZ echantillonnees apres centrage et division par `scale`. |
| `normal` | Normale 3D associee a chaque point echantillonne. |
| `normal_source` | Origine des normales: lues depuis l'OBJ ou calculees depuis les faces. |
| `curvature` | Courbure moyenne normalisee par point, ou `None` si `CURVATURE=False`. |
| `y_binary` | Label binaire par point: `0` hors dent, `1` dent. |
| `y_fdi` | Label FDI original par point; `0` represente le fond / non-dent. |
| `y_fdi_class` | Label FDI remappe vers des classes contigues pour l'entrainement. |
| `y_instance` | Identifiant d'instance dentaire par point; `0` represente le fond / non-dent. |
| `fdi_to_class` | Dictionnaire de conversion FDI -> classe contigue. |
| `class_to_fdi` | Dictionnaire inverse classe contigue -> FDI. |
| `source_indices` | Indices des sommets originaux conserves apres echantillonnage. |
| `landmarks_raw` | Landmarks en coordonnees originales, groupes par dent FDI et par classe. |
| `landmarks_norm` | Landmarks en coordonnees normalisees, groupes par dent FDI et par classe. |
| `landmark_to_tooth` | Liste plate reliant chaque landmark a sa dent, son sommet proche et son index echantillonne si present. |
| `tooth_centers_raw` | Centre moyen de chaque dent en coordonnees originales. |
| `tooth_centers_norm` | Centre moyen de chaque dent transforme dans le repere normalise. |
| `center` | Centre du mesh utilise pour normaliser les coordonnees. |
| `scale` | Rayon max utilise pour remettre le mesh dans un repere de rayon 1. |